# Building Agents with LangGraph Course #6: A Guide to Human-in-the-Loop Interactions

Companion notebook for [the complete To Data & Beyond tutorial](https://todatabeyond.com/blog/building-agents-with-langgraph-course-6-a-guide-to-human-in-the-loop-interactions). View the [maintained notebook on GitHub](https://github.com/To-Data-Beyond/Generative-AI-Techanical-Tutorials/blob/main/LangGraph_Course_6_Human_in_the_Loop.ipynb).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/To-Data-Beyond/Generative-AI-Techanical-Tutorials/blob/main/LangGraph_Course_6_Human_in_the_Loop.ipynb)


## Before you begin

Add provider keys through Colab secrets or environment variables. Run cells in order and inspect every interrupt before approving a tool action.

This notebook intentionally contains no saved execution outputs or credentials.


## Environment setup


In [ ]:
!pip install -q langgraph langchain-openai langchain-community tavily-python python-dotenv langgraph-checkpoint-sqlite aiosqlite


Welcome back to our Building Agents with LangGraph series! In the previous articles, we’ve constructed a capable agent that can use tools to answer questions.

However, in many real-world scenarios, we need more control. We might want to approve an agent’s actions before it executes them, correct its course if it misunderstands, or even explore alternative paths.

This is where the concept of “Human in the Loop” (HITL) becomes essential. LangGraph is designed with this interactivity in mind, providing powerful tools for pausing, inspecting, and manipulating the state of your agent.

In this article, we will explore these advanced human-in-the-loop patterns:

- Manual Approval: How to interrupt the graph’s execution before a critical step, like a tool call, to allow for human review.
- State Modification: How to directly edit the agent’s state to correct its actions or steer its behavior.
- Time Travel: A fascinating feature that lets you rewind to any previous state in the conversation and branch off to explore a new path.
- Injecting Tool Outputs: How to manually provide tool results to the agent, bypassing the actual tool execution.

Let’s dive in and see how to assign a human supervisor to our agent.

## Table of Contents

This article is the Sixth Article in the ongoing series of Building LLM Agents with LangGraph:

- Introduction to Agents & LangGraph (Published!)
- Building Simple ReAct Agent from Scratch (Published!)
- Main Building Units of LangGraph (Published!)
- Agentic Search Tools in LangGraph (Published!)
- Persistence and Streaming in LangGraph (Published!)
- Human in the Loop in LLM Agents (You are here!)
- Putting it All Together! Building Essay Writer Agent (Coming Soon!)

This series is designed to take readers from foundational knowledge to advanced practices in building LLM agents with LangGraph.

Each article delves into essential components, such as constructing simple ReAct agents from scratch, leveraging LangGraph’s building units, utilizing agentic search tools, implementing persistence and streaming capabilities, integrating human-in-the-loop interactions, and culminating in the creation of a fully functional essay-writing agent.

By the end of this series, you will have a comprehensive understanding of LangGraph, practical skills to design and deploy LLM agents, and the confidence to build customized AI-driven workflows tailored to diverse applications.

---

## 1. Project Setup and Dependencies

We’ll begin with the familiar setup from our previous article. First, we configure our environment variables and import the necessary libraries. We’ll use SqliteSaver to enable checkpointing, which is the mechanism that saves our graph’s state and makes all these interactive patterns possible.


In [ ]:
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.checkpoint.sqlite import SqliteSaver

_ = load_dotenv()


memory = SqliteSaver.from_conn_string(":memory:")


For human-in-the-loop interactions, we often need to modify or replace messages in our agent’s state. The standard operator.add() reducer simply appends messages. To gain more control, we’ll create a custom reducer function, reduce_messages.

This function will inspect incoming messages. If a new message has the same unique ID as an existing one, it will replace it. Otherwise, it will append the new message. This allows us to overwrite previous steps if needed.


In [ ]:
from uuid import uuid4
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage

"""
In previous examples we've annotated the `messages` state key
with the default `operator.add` or `+` reducer, which always
appends new messages to the end of the existing messages array.

Now, to support replacing existing messages, we annotate the
`messages` key with a customer reducer function, which replaces
messages with the same `id`, and appends them otherwise.
"""
def reduce_messages(left: list[AnyMessage], right: list[AnyMessage]) -> list[AnyMessage]:
    # assign ids to messages that don't have them
    for message in right:
        if not message.id:
            message.id = str(uuid4())
    # merge the new messages with the existing messages
    merged = left.copy()
    for message in right:
        for i, existing in enumerate(merged):
            # replace any existing messages with the same id
            if existing.id == message.id:
                merged[i] = message
                break
        else:
            # append any new messages to the end
            merged.append(message)
    return merged

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], reduce_messages]


We’ll use the same TavilySearchResults tool as before.


In [ ]:
tool = TavilySearchResults(max_results=2)


---

## 2. Interrupting Execution for Manual Approval

Now, let’s build our agent. The core logic remains the same, but we’ll introduce one crucial change during the compilation step. By adding interrupt_before=[“action”], we instruct LangGraph to pause the execution right before calling the action node. Since our action node is responsible for executing tools, this effectively creates an approval gate.


In [ ]:
class Agent:
    def __init__(self, model, tools, system="", checkpointer=None):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(
            checkpointer=checkpointer,
            # This is the new parameter we're adding
            interrupt_before=["action"]
        )
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        print(state)
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}


Let’s initialize the agent and make a request.


In [ ]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatOpenAI(model="gpt-3.5-turbo")
abot = Agent(model, [tool], system=prompt, checkpointer=memory)


In [ ]:
messages = [HumanMessage(content="Whats the weather in SF?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)


**Expected output**

```text
{‘messages’: [HumanMessage(content=’Whats the weather in SF?’, id=’0d8ac19e-6fc1–406f-924d-9ddcc87b8e29'), AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘id’: ‘call_i7rGhnzgZf5hW3bsDcqdTrH0’, ‘function’: {‘arguments’: ‘{“query”:”weather in San Francisco”}’, ‘name’: ‘tavily_search_results_json’}, ‘type’: ‘function’}]}, response_metadata={‘token_usage’: {‘completion_tokens’: 22, ‘prompt_tokens’: 152, ‘total_tokens’: 174, ‘prompt_tokens_details’: {‘cached_tokens’: 0, ‘audio_tokens’: 0}, ‘completion_tokens_details’: {‘reasoning_tokens’: 0, ‘audio_tokens’: 0, ‘accepted_prediction_tokens’: 0, ‘rejected_prediction_tokens’: 0}}, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘finish_reason’: ‘tool_calls’, ‘logprobs’: None}, id=’run-ef442ef3–7e27–4d37-b0f4-ba29d1161590–0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘weather in San Francisco’}, ‘id’: ‘call_i7rGhnzgZf5hW3bsDcqdTrH0’}])]}
{‘messages’: [AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘id’: ‘call_i7rGhnzgZf5hW3bsDcqdTrH0’, ‘function’: {‘arguments’: ‘{“query”:”weather in San Francisco”}’, ‘name’: ‘tavily_search_results_json’}, ‘type’: ‘function’}]}, response_metadata={‘token_usage’: {‘completion_tokens’: 22, ‘prompt_tokens’: 152, ‘total_tokens’: 174, ‘prompt_tokens_details’: {‘cached_tokens’: 0, ‘audio_tokens’: 0}, ‘completion_tokens_details’: {‘reasoning_tokens’: 0, ‘audio_tokens’: 0, ‘accepted_prediction_tokens’: 0, ‘rejected_prediction_tokens’: 0}}, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘finish_reason’: ‘tool_calls’, ‘logprobs’: None}, id=’run-ef442ef3–7e27–4d37-b0f4-ba29d1161590–0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘weather in San Francisco’}, ‘id’: ‘call_i7rGhnzgZf5hW3bsDcqdTrH0’}])]}
```


Notice the output stops after the AIMessage containing the tool_calls. The graph has paused because of our interrupt_before configuration. We can inspect the current state to confirm this.


In [ ]:
abot.graph.get_state(thread)


**Expected output**

```text
StateSnapshot(values={‘messages’: [HumanMessage(content=’Whats the weather in SF?’, id=’0d8ac19e-6fc1–406f-924d-9ddcc87b8e29'), AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘function’: {‘arguments’: ‘{“query”:”weather in San Francisco”}’, ‘name’: ‘tavily_search_results_json’}, ‘id’: ‘call_i7rGhnzgZf5hW3bsDcqdTrH0’, ‘type’: ‘function’}]}, response_metadata={‘finish_reason’: ‘tool_calls’, ‘logprobs’: None, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘token_usage’: {‘completion_tokens’: 22, ‘completion_tokens_details’: {‘accepted_prediction_tokens’: 0, ‘audio_tokens’: 0, ‘reasoning_tokens’: 0, ‘rejected_prediction_tokens’: 0}, ‘prompt_tokens’: 152, ‘prompt_tokens_details’: {‘audio_tokens’: 0, ‘cached_tokens’: 0}, ‘total_tokens’: 174}}, id=’run-ef442ef3–7e27–4d37-b0f4-ba29d1161590–0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘weather in San Francisco’}, ‘id’: ‘call_i7rGhnzgZf5hW3bsDcqdTrH0’}])]}, next=(‘action’,), config={‘configurable’: {‘thread_id’: ‘1’, ‘thread_ts’: ‘1f07adc9–50ca-6c39–8001–98c519bd55d6’}}, metadata={‘source’: ‘loop’, ‘step’: 1, ‘writes’: {‘llm’: {‘messages’: [AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘function’: {‘arguments’: ‘{“query”:”weather in San Francisco”}’, ‘name’: ‘tavily_search_results_json’}, ‘id’: ‘call_i7rGhnzgZf5hW3bsDcqdTrH0’, ‘type’: ‘function’}]}, response_metadata={‘finish_reason’: ‘tool_calls’, ‘logprobs’: None, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘token_usage’: {‘completion_tokens’: 22, ‘completion_tokens_details’: {‘accepted_prediction_tokens’: 0, ‘audio_tokens’: 0, ‘reasoning_tokens’: 0, ‘rejected_prediction_tokens’: 0}, ‘prompt_tokens’: 152, ‘prompt_tokens_details’: {‘audio_tokens’: 0, ‘cached_tokens’: 0}, ‘total_tokens’: 174}}, id=’run-ef442ef3–7e27–4d37-b0f4-ba29d1161590–0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘weather in San Francisco’}, ‘id’: ‘call_i7rGhnzgZf5hW3bsDcqdTrH0’}])]}}}, created_at=’2025–08–16T20:07:06.051464+00:00', parent_config={‘configurable’: {‘thread_id’: ‘1’, ‘thread_ts’: ‘1f07adc9–505b-6a71–8000–30a74372b034’}})
```


The output of get_state is a StateSnapshot object. Look for the next attribute.


In [ ]:
abot.graph.get_state(thread).next


**Expected output**

```text
(‘action’,)
```


The output (‘action’,) confirms that the graph is paused and waiting to execute the action node.

To continue, we simply stream again, passing None as the input but using the same thread configuration. This tells LangGraph to pick up where it left off.


In [ ]:
for event in abot.graph.stream(None, thread):
    for v in event.values():
        print(v)


**Expected output**

```text
Calling: {‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘weather in San Francisco’}, ‘id’: ‘call_i7rGhnzgZf5hW3bsDcqdTrH0’}
Back to the model!
{‘messages’: [ToolMessage(content=”[{‘url’: ‘https://en.climate-data.org/north-america/united-states-of-america/california/san-francisco-385/t/august-8/', ‘content’: ‘| 18. August | 17 °C | 62 °F | 22 °C | 72 °F | 13 °C | 56 °F | 14 °C | 57 °F | 0.0 mm | 0.0 inch. |\\n| 19. August | 17 °C | 62 °F | 22 °C | 72 °F | 13 °C | 56 °F | 14 °C | 57 °F | 0.1 mm | 0.0 inch. |\\n| 20. August | 17 °C | 62 °F | 21 °C | 71 °F | 13 °C | 56 °F | 14 °C | 57 °F | 1.0 mm | 0.0 inch. |\\n| 21. August | 17 °C | 62 °F | 22 °C | 72 °F | 13 °C | 56 °F | 14 °C | 57 °F | 0.1 mm | 0.0 inch. |\\n| 22. August | 17 °C | 63 °F | 23 °C | 73 °F | 14 °C | 57 °F | 14 °C | 57 °F | 0.0 mm | 0.0 inch. | […] | Max. Temperature °C (°F) | 14 °C (57.3) °F | 14.9 °C (58.7) °F | 16.2 °C (61.2) °F | 17.4 °C (63.3) °F | 19.2 °C (66.5) °F | 21.5 °C (70.8) °F | 21.8 °C (71.2) °F | 22.2 °C (71.9) °F | 23.1 °C (73.6) °F | 21.3 °C (70.3) °F | 17.1 °C (62.8) °F | 13.9 °C (57.1) °F |\\n| Precipitation / Rainfall mm (in) | 113 (4) | 118 (4) | 83 (3) | 40 (1) | 21 (0) | 6 (0) | 2 (0) | 2 (0) | 3 (0) | 25 (0) | 57 (2) | 111 (4) | […] | Min. Temperature °C (°F) | 6.2 °C (43.2) °F | 7.1 °C (44.8) °F | 8.2 °C (46.8) °F | 8.9 °C (48.1) °F | 10.3 °C (50.6) °F | 11.8 °C (53.3) °F | 12.7 °C (54.9) °F | 13.3 °C (55.9) °F | 13.1 °C (55.6) °F | 11.9 °C (53.4) °F | 9 °C (48.2) °F | 6.8 °C (44.2) °F |’}, {‘url’: ‘https://www.weather2travel.com/california/san-francisco/august/', ‘content’: ‘weather2travel.com — travel deals for your holiday in the sun\\nClick to search\\n\\n# San Francisco weather in August 2025\\n\\nExpect daytime maximum temperatures of 20°C in San Francisco, California in August based on long-term weather averages. There are 10 hours of sunshine per day on average.’}]”, name=’tavily_search_results_json’, id=’bd1cad08–729b-46af-a8db-6fbb66d2b7c1', tool_call_id=’call_i7rGhnzgZf5hW3bsDcqdTrH0')]}
{‘messages’: [HumanMessage(content=’Whats the weather in SF?’, id=’d3cafb23–9573–443e-b530–512bb6e6b4f1'), AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘function’: {‘arguments’: ‘{“query”:”weather in San Francisco”}’, ‘name’: ‘tavily_search_results_json’}, ‘id’: ‘call_i7rGhnzgZf5hW3bsDcqdTrH0’, ‘type’: ‘function’}]}, response_metadata={‘finish_reason’: ‘tool_calls’, ‘logprobs’: None, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘token_usage’: {‘completion_tokens’: 22, ‘completion_tokens_details’: {‘accepted_prediction_tokens’: 0, ‘audio_tokens’: 0, ‘reasoning_tokens’: 0, ‘rejected_prediction_tokens’: 0}, ‘prompt_tokens’: 152, ‘prompt_tokens_details’: {‘audio_tokens’: 0, ‘cached_tokens’: 0}, ‘total_tokens’: 174}}, id=’run-01d73196-ce30–4068-b5a6–484bafd0aeee-0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘weather in San Francisco’}, ‘id’: ‘call_i7rGhnzgZf5hW3bsDcqdTrH0’}]), ToolMessage(content=”[{‘url’: ‘https://en.climate-data.org/north-america/united-states-of-america/california/san-francisco-385/t/august-8/', ‘content’: ‘| 18. August | 17 °C | 62 °F | 22 °C | 72 °F | 13 °C | 56 °F | 14 °C | 57 °F | 0.0 mm | 0.0 inch. |\\n| 19. August | 17 °C | 62 °F | 22 °C | 72 °F | 13 °C | 56 °F | 14 °C | 57 °F | 0.1 mm | 0.0 inch. |\\n| 20. August | 17 °C | 62 °F | 21 °C | 71 °F | 13 °C | 56 °F | 14 °C | 57 °F | 1.0 mm | 0.0 inch. |\\n| 21. August | 17 °C | 62 °F | 22 °C | 72 °F | 13 °C | 56 °F | 14 °C | 57 °F | 0.1 mm | 0.0 inch. |\\n| 22. August | 17 °C | 63 °F | 23 °C | 73 °F | 14 °C | 57 °F | 14 °C | 57 °F | 0.0 mm | 0.0 inch. | […] | Max. Temperature °C (°F) | 14 °C (57.3) °F | 14.9 °C (58.7) °F | 16.2 °C (61.2) °F | 17.4 °C (63.3) °F | 19.2 °C (66.5) °F | 21.5 °C (70.8) °F | 21.8 °C (71.2) °F | 22.2 °C (71.9) °F | 23.1 °C (73.6) °F | 21.3 °C (70.3) °F | 17.1 °C (62.8) °F | 13.9 °C (57.1) °F |\\n| Precipitation / Rainfall mm (in) | 113 (4) | 118 (4) | 83 (3) | 40 (1) | 21 (0) | 6 (0) | 2 (0) | 2 (0) | 3 (0) | 25 (0) | 57 (2) | 111 (4) | […] | Min. Temperature °C (°F) | 6.2 °C (43.2) °F | 7.1 °C (44.8) °F | 8.2 °C (46.8) °F | 8.9 °C (48.1) °F | 10.3 °C (50.6) °F | 11.8 °C (53.3) °F | 12.7 °C (54.9) °F | 13.3 °C (55.9) °F | 13.1 °C (55.6) °F | 11.9 °C (53.4) °F | 9 °C (48.2) °F | 6.8 °C (44.2) °F |’}, {‘url’: ‘https://www.weather2travel.com/california/san-francisco/august/', ‘content’: ‘weather2travel.com — travel deals for your holiday in the sun\\nClick to search\\n\\n# San Francisco weather in August 2025\\n\\nExpect daytime maximum temperatures of 20°C in San Francisco, California in August based on long-term weather averages. There are 10 hours of sunshine per day on average.’}]”, name=’tavily_search_results_json’, id=’bd1cad08–729b-46af-a8db-6fbb66d2b7c1', tool_call_id=’call_i7rGhnzgZf5hW3bsDcqdTrH0'), AIMessage(content=’The weather in San Francisco, California in August typically has daytime maximum temperatures around 20°C (68°F) based on long-term weather averages. There are about 10 hours of sunshine per day on average.’, response_metadata={‘token_usage’: {‘completion_tokens’: 43, ‘prompt_tokens’: 1097, ‘total_tokens’: 1140, ‘prompt_tokens_details’: {‘cached_tokens’: 0, ‘audio_tokens’: 0}, ‘completion_tokens_details’: {‘reasoning_tokens’: 0, ‘audio_tokens’: 0, ‘accepted_prediction_tokens’: 0, ‘rejected_prediction_tokens’: 0}}, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘finish_reason’: ‘stop’, ‘logprobs’: None}, id=’run-db559cf6–03b6–4849-ac39-c6db47fece82–0')]}
{‘messages’: [AIMessage(content=’The weather in San Francisco, California in August typically has daytime maximum temperatures around 20°C (68°F) based on long-term weather averages. There are about 10 hours of sunshine per day on average.’, response_metadata={‘token_usage’: {‘completion_tokens’: 43, ‘prompt_tokens’: 1097, ‘total_tokens’: 1140, ‘prompt_tokens_details’: {‘cached_tokens’: 0, ‘audio_tokens’: 0}, ‘completion_tokens_details’: {‘reasoning_tokens’: 0, ‘audio_tokens’: 0, ‘accepted_prediction_tokens’: 0, ‘rejected_prediction_tokens’: 0}}, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘finish_reason’: ‘stop’, ‘logprobs’: None}, id=’run-db559cf6–03b6–4849-ac39-c6db47fece82–0')]}
```


This time, the execution completes. The agent calls the tool and generates the final response. If we check the next node now, it will be empty, indicating the graph has finished.


In [ ]:
abot.graph.get_state(thread).next
# Output: ()


**Expected output**

```text
()
```


We can even wrap this in a simple loop to create an interactive command-line approval process.


In [ ]:
messages = [HumanMessage("Whats the weather in LA?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)
while abot.graph.get_state(thread).next:
    print("\n", abot.graph.get_state(thread),"\n")
    _input = input("proceed?")
    if _input != "y":
        print("aborting")
        break
    for event in abot.graph.stream(None, thread):
        for v in event.values():
            print(v)


**Expected output**

```text
{‘messages’: [HumanMessage(content=’Whats the weather in LA?’, id=’64073328–247e-4d53–8f67–908b59f20a62'), AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’, ‘function’: {‘arguments’: ‘{“query”:”weather in Los Angeles”}’, ‘name’: ‘tavily_search_results_json’}, ‘type’: ‘function’}]}, response_metadata={‘token_usage’: {‘completion_tokens’: 22, ‘prompt_tokens’: 152, ‘total_tokens’: 174, ‘prompt_tokens_details’: {‘cached_tokens’: 0, ‘audio_tokens’: 0}, ‘completion_tokens_details’: {‘reasoning_tokens’: 0, ‘audio_tokens’: 0, ‘accepted_prediction_tokens’: 0, ‘rejected_prediction_tokens’: 0}}, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘finish_reason’: ‘tool_calls’, ‘logprobs’: None}, id=’run-95861c64-f69d-444f-8b1e-bc2da097f7ec-0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘weather in Los Angeles’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}])]}
{‘messages’: [AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’, ‘function’: {‘arguments’: ‘{“query”:”weather in Los Angeles”}’, ‘name’: ‘tavily_search_results_json’}, ‘type’: ‘function’}]}, response_metadata={‘token_usage’: {‘completion_tokens’: 22, ‘prompt_tokens’: 152, ‘total_tokens’: 174, ‘prompt_tokens_details’: {‘cached_tokens’: 0, ‘audio_tokens’: 0}, ‘completion_tokens_details’: {‘reasoning_tokens’: 0, ‘audio_tokens’: 0, ‘accepted_prediction_tokens’: 0, ‘rejected_prediction_tokens’: 0}}, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘finish_reason’: ‘tool_calls’, ‘logprobs’: None}, id=’run-95861c64-f69d-444f-8b1e-bc2da097f7ec-0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘weather in Los Angeles’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}])]}
```


**Expected output**

```text
StateSnapshot(values={‘messages’: [HumanMessage(content=’Whats the weather in LA?’, id=’64073328–247e-4d53–8f67–908b59f20a62'), AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘function’: {‘arguments’: ‘{“query”:”weather in Los Angeles”}’, ‘name’: ‘tavily_search_results_json’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’, ‘type’: ‘function’}]}, response_metadata={‘finish_reason’: ‘tool_calls’, ‘logprobs’: None, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘token_usage’: {‘completion_tokens’: 22, ‘completion_tokens_details’: {‘accepted_prediction_tokens’: 0, ‘audio_tokens’: 0, ‘reasoning_tokens’: 0, ‘rejected_prediction_tokens’: 0}, ‘prompt_tokens’: 152, ‘prompt_tokens_details’: {‘audio_tokens’: 0, ‘cached_tokens’: 0}, ‘total_tokens’: 174}}, id=’run-95861c64-f69d-444f-8b1e-bc2da097f7ec-0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘weather in Los Angeles’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}])]}, next=(‘action’,), config={‘configurable’: {‘thread_id’: ‘2’, ‘thread_ts’: ‘1f07be88-b98c-6ebc-8001-dcc406d8f0f0’}}, metadata={‘source’: ‘loop’, ‘step’: 1, ‘writes’: {‘llm’: {‘messages’: [AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘function’: {‘arguments’: ‘{“query”:”weather in Los Angeles”}’, ‘name’: ‘tavily_search_results_json’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’, ‘type’: ‘function’}]}, response_metadata={‘finish_reason’: ‘tool_calls’, ‘logprobs’: None, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘token_usage’: {‘completion_tokens’: 22, ‘completion_tokens_details’: {‘accepted_prediction_tokens’: 0, ‘audio_tokens’: 0, ‘reasoning_tokens’: 0, ‘rejected_prediction_tokens’: 0}, ‘prompt_tokens’: 152, ‘prompt_tokens_details’: {‘audio_tokens’: 0, ‘cached_tokens’: 0}, ‘total_tokens’: 174}}, id=’run-95861c64-f69d-444f-8b1e-bc2da097f7ec-0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘weather in Los Angeles’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}])]}}}, created_at=’2025–08–18T04:05:15.316172+00:00', parent_config={‘configurable’: {‘thread_id’: ‘2’, ‘thread_ts’: ‘1f07be88-b941–6e49–8000-ad365b4fd777’}})
```


**Expected output**

```text
proceed?y
Calling: {‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘weather in Los Angeles’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}
Back to the model!
{‘messages’: [ToolMessage(content=”[{‘url’: ‘https://weathershogun.com/weather/usa/ca/los-angeles/451/august/2025-08-18', ‘content’: ‘Monday, August 18, 2025. Los Angeles, CA — Weather Forecast \\n\\n===============\\n\\n☰\\n\\nLos Angeles, CA\\n\\nImage 1: WeatherShogun.com\\n\\nHomeContactBrowse StatesPrivacy PolicyTerms and Conditions\\n\\n°F)°C)\\n\\n❮\\n\\nTodayTomorrowHourly7 days30 daysAugust\\n\\n❯\\n\\nLos Angeles, California Weather: \\n\\nMonday, August 18, 2025\\n\\nDay 84°\\n\\nNight 64°\\n\\nPrecipitation 0 %\\n\\nWind 8 mph\\n\\nUV Index (0–11+)11\\n\\nTuesday\\n\\n Hourly\\n Today\\n Current Air Quality\\n Hourly Air Quality Forecast\\n 7 days\\n 30 days’}, {‘url’: ‘https://en.climate-data.org/north-america/united-states-of-america/california/los-angeles-714829/t/august-8/', ‘content’: ‘| Max. Temperature °C (°F) | 19.5 °C (67.2) °F | 19.4 °C (66.9) °F | 21.4 °C (70.5) °F | 23.3 °C (73.9) °F | 25.2 °C (77.4) °F | 28.1 °C (82.6) °F | 31.3 °C (88.3) °F | 31.9 °C (89.5) °F | 31 °C (87.7) °F | 27.2 °C (81) °F | 23.1 °C (73.5) °F | 18.8 °C (65.9) °F |\\n| Precipitation / Rainfall mm (in) | 84 (3) | 89 (3) | 54 (2) | 19 (0) | 11 (0) | 3 (0) | 2 (0) | 0 (0) | 4 (0) | 17 (0) | 21 (0) | 53 (2) | […] | Humidity(%) | 52% | 57% | 59% | 55% | 56% | 55% | 52% | 49% | 49% | 49% | 46% | 53% |\\n| Rainy days (d) | 4 | 5 | 4 | 2 | 1 | 0 | 0 | 0 | 1 | 2 | 2 | 4 |\\n| avg. Sun hours (hours) | 7.6 | 7.6 | 8.3 | 9.0 | 9.1 | 10.2 | 11.3 | 10.8 | 9.5 | 8.3 | 7.9 | 7.4 | […] ## \\n\\n## \\n\\n# Los Angeles Weather in August\\n\\nAre you planning a holiday with hopefully nice weather in Los Angeles in August 2025? Here you can find all information about the weather in Los Angeles in August:\\n\\n## Los Angeles weather in August\\n\\n| | | | | | |\\n| — — | — — | — — | — — | — — | — — |\\n| | Temperature August | 24.5°C | 76.1°F | | Precipitation / Rainfall August | 0mm | 0 inches |\\n| | Temperature August max. | 31.9°C | 89.5°F |\\n| | Temperature August min. | 18.2°C | 64.8°F |’}]”, name=’tavily_search_results_json’, id=’a1f1c6b0–198a-42b8–9323–065007924151', tool_call_id=’call_6ED1ZQ8nrjYIOY14yqInLPZc’)]}
{‘messages’: [HumanMessage(content=’Whats the weather in LA?’, id=’64073328–247e-4d53–8f67–908b59f20a62'), AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘function’: {‘arguments’: ‘{“query”:”weather in Los Angeles”}’, ‘name’: ‘tavily_search_results_json’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’, ‘type’: ‘function’}]}, response_metadata={‘finish_reason’: ‘tool_calls’, ‘logprobs’: None, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘token_usage’: {‘completion_tokens’: 22, ‘completion_tokens_details’: {‘accepted_prediction_tokens’: 0, ‘audio_tokens’: 0, ‘reasoning_tokens’: 0, ‘rejected_prediction_tokens’: 0}, ‘prompt_tokens’: 152, ‘prompt_tokens_details’: {‘audio_tokens’: 0, ‘cached_tokens’: 0}, ‘total_tokens’: 174}}, id=’run-95861c64-f69d-444f-8b1e-bc2da097f7ec-0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘weather in Los Angeles’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}]), ToolMessage(content=”[{‘url’: ‘https://weathershogun.com/weather/usa/ca/los-angeles/451/august/2025-08-18', ‘content’: ‘Monday, August 18, 2025. Los Angeles, CA — Weather Forecast \\n\\n===============\\n\\n☰\\n\\nLos Angeles, CA\\n\\nImage 1: WeatherShogun.com\\n\\nHomeContactBrowse StatesPrivacy PolicyTerms and Conditions\\n\\n°F)°C)\\n\\n❮\\n\\nTodayTomorrowHourly7 days30 daysAugust\\n\\n❯\\n\\nLos Angeles, California Weather: \\n\\nMonday, August 18, 2025\\n\\nDay 84°\\n\\nNight 64°\\n\\nPrecipitation 0 %\\n\\nWind 8 mph\\n\\nUV Index (0–11+)11\\n\\nTuesday\\n\\n Hourly\\n Today\\n Current Air Quality\\n Hourly Air Quality Forecast\\n 7 days\\n 30 days’}, {‘url’: ‘https://en.climate-data.org/north-america/united-states-of-america/california/los-angeles-714829/t/august-8/', ‘content’: ‘| Max. Temperature °C (°F) | 19.5 °C (67.2) °F | 19.4 °C (66.9) °F | 21.4 °C (70.5) °F | 23.3 °C (73.9) °F | 25.2 °C (77.4) °F | 28.1 °C (82.6) °F | 31.3 °C (88.3) °F | 31.9 °C (89.5) °F | 31 °C (87.7) °F | 27.2 °C (81) °F | 23.1 °C (73.5) °F | 18.8 °C (65.9) °F |\\n| Precipitation / Rainfall mm (in) | 84 (3) | 89 (3) | 54 (2) | 19 (0) | 11 (0) | 3 (0) | 2 (0) | 0 (0) | 4 (0) | 17 (0) | 21 (0) | 53 (2) | […] | Humidity(%) | 52% | 57% | 59% | 55% | 56% | 55% | 52% | 49% | 49% | 49% | 46% | 53% |\\n| Rainy days (d) | 4 | 5 | 4 | 2 | 1 | 0 | 0 | 0 | 1 | 2 | 2 | 4 |\\n| avg. Sun hours (hours) | 7.6 | 7.6 | 8.3 | 9.0 | 9.1 | 10.2 | 11.3 | 10.8 | 9.5 | 8.3 | 7.9 | 7.4 | […] ## \\n\\n## \\n\\n# Los Angeles Weather in August\\n\\nAre you planning a holiday with hopefully nice weather in Los Angeles in August 2025? Here you can find all information about the weather in Los Angeles in August:\\n\\n## Los Angeles weather in August\\n\\n| | | | | | |\\n| — — | — — | — — | — — | — — | — — |\\n| | Temperature August | 24.5°C | 76.1°F | | Precipitation / Rainfall August | 0mm | 0 inches |\\n| | Temperature August max. | 31.9°C | 89.5°F |\\n| | Temperature August min. | 18.2°C | 64.8°F |’}]”, name=’tavily_search_results_json’, id=’a1f1c6b0–198a-42b8–9323–065007924151', tool_call_id=’call_6ED1ZQ8nrjYIOY14yqInLPZc’), AIMessage(content=’The weather in Los Angeles today is 84°F during the day and 64°F at night. There is no precipitation expected with a wind speed of 8 mph and a UV Index of 11.’, response_metadata={‘token_usage’: {‘completion_tokens’: 42, ‘prompt_tokens’: 1065, ‘total_tokens’: 1107, ‘prompt_tokens_details’: {‘cached_tokens’: 0, ‘audio_tokens’: 0}, ‘completion_tokens_details’: {‘reasoning_tokens’: 0, ‘audio_tokens’: 0, ‘accepted_prediction_tokens’: 0, ‘rejected_prediction_tokens’: 0}}, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘finish_reason’: ‘stop’, ‘logprobs’: None}, id=’run-ed2612c2–8e62–4664-b4c1-d6572317d3bc-0')]}
{‘messages’: [AIMessage(content=’The weather in Los Angeles today is 84°F during the day and 64°F at night. There is no precipitation expected with a wind speed of 8 mph and a UV Index of 11.’, response_metadata={‘token_usage’: {‘completion_tokens’: 42, ‘prompt_tokens’: 1065, ‘total_tokens’: 1107, ‘prompt_tokens_details’: {‘cached_tokens’: 0, ‘audio_tokens’: 0}, ‘completion_tokens_details’: {‘reasoning_tokens’: 0, ‘audio_tokens’: 0, ‘accepted_prediction_tokens’: 0, ‘rejected_prediction_tokens’: 0}}, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘finish_reason’: ‘stop’, ‘logprobs’: None}, id=’run-ed2612c2–8e62–4664-b4c1-d6572317d3bc-0')]}
```


---

## 3. Understanding State Memory and Time Travel

Before we start modifying state, let’s understand how LangGraph’s memory works. As a graph executes, the checkpointer saves a snapshot of the state at every step. Each snapshot contains:

1. values: The actual AgentState (e.g., our list of messages).
2. config: A configuration dictionary containing the thread_id and a unique timestamp, thread_ts. This thread_ts is the key to time travel, as it uniquely identifies a specific state snapshot.

You can interact with this memory using several methods:

- get_state(config): Retrieves a specific snapshot. If you only provide the thread_id, it returns the latest state.
- get_state_history(config): Returns an iterator over all snapshots for a given thread, from newest to oldest.
- update_state(config, values): Creates a new state snapshot by modifying an existing one.

This architecture is what enables “time travel.” By retrieving the config of a past state, we can use it to resume execution from that exact point, effectively rewinding the agent’s process.

### 3.1. Modifying State on the Fly

Let’s see this in action. Suppose our agent misinterprets “LA” as “Los Angeles” when we meant “Louisiana.” We can intervene and correct it.

First, we start a new thread and run it until the interrupt.


In [ ]:
messages = [HumanMessage("Whats the weather in LA?")]
thread = {"configurable": {"thread_id": "3"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)


**Expected output**

```text
{‘messages’: [HumanMessage(content=’Whats the weather in LA?’, id=’abe1c1c0-f305–479d-b788–820269a138ce’), AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’, ‘function’: {‘arguments’: ‘{“query”:”weather in Los Angeles”}’, ‘name’: ‘tavily_search_results_json’}, ‘type’: ‘function’}]}, response_metadata={‘token_usage’: {‘completion_tokens’: 22, ‘prompt_tokens’: 152, ‘total_tokens’: 174, ‘prompt_tokens_details’: {‘cached_tokens’: 0, ‘audio_tokens’: 0}, ‘completion_tokens_details’: {‘reasoning_tokens’: 0, ‘audio_tokens’: 0, ‘accepted_prediction_tokens’: 0, ‘rejected_prediction_tokens’: 0}}, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘finish_reason’: ‘tool_calls’, ‘logprobs’: None}, id=’run-7ac75989–7448–4d63-a807–49a0061317ae-0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘weather in Los Angeles’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}])]}
{‘messages’: [AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’, ‘function’: {‘arguments’: ‘{“query”:”weather in Los Angeles”}’, ‘name’: ‘tavily_search_results_json’}, ‘type’: ‘function’}]}, response_metadata={‘token_usage’: {‘completion_tokens’: 22, ‘prompt_tokens’: 152, ‘total_tokens’: 174, ‘prompt_tokens_details’: {‘cached_tokens’: 0, ‘audio_tokens’: 0}, ‘completion_tokens_details’: {‘reasoning_tokens’: 0, ‘audio_tokens’: 0, ‘accepted_prediction_tokens’: 0, ‘rejected_prediction_tokens’: 0}}, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘finish_reason’: ‘tool_calls’, ‘logprobs’: None}, id=’run-7ac75989–7448–4d63-a807–49a0061317ae-0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘weather in Los Angeles’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}])]}
```


Now, let’s grab the current state and inspect the tool call it’s about to make.


In [ ]:
current_values = abot.graph.get_state(thread)
current_values.values['messages'][-1].tool_calls


**Expected output**

```text
[{‘name’: ‘tavily_search_results_json’,
 ‘args’: {‘query’: ‘weather in Los Angeles’},
 ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}]
```


We can see it’s about to search for Los Angeles. Let’s change that. We’ll modify the `args` in the `tool_calls` list to point to Louisiana instead.


In [ ]:
_id = current_values.values['messages'][-1].tool_calls[0]['id']
current_values.values['messages'][-1].tool_calls = [
    {'name': 'tavily_search_results_json',
     'args': {'query': 'current weather in Louisiana'},
     'id': _id}
]


This modification is just local for now. To commit it back to the graph’s memory, we use update_state.


In [ ]:
abot.graph.update_state(thread, current_values.values)


**Expected output**

```text
{‘messages’: [HumanMessage(content=’Whats the weather in LA?’, id=’abe1c1c0-f305–479d-b788–820269a138ce’), AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘function’: {‘arguments’: ‘{“query”:”weather in Los Angeles”}’, ‘name’: ‘tavily_search_results_json’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’, ‘type’: ‘function’}]}, response_metadata={‘finish_reason’: ‘tool_calls’, ‘logprobs’: None, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘token_usage’: {‘completion_tokens’: 22, ‘completion_tokens_details’: {‘accepted_prediction_tokens’: 0, ‘audio_tokens’: 0, ‘reasoning_tokens’: 0, ‘rejected_prediction_tokens’: 0}, ‘prompt_tokens’: 152, ‘prompt_tokens_details’: {‘audio_tokens’: 0, ‘cached_tokens’: 0}, ‘total_tokens’: 174}}, id=’run-7ac75989–7448–4d63-a807–49a0061317ae-0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘current weather in Louisiana’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}])]}
{‘configurable’: {‘thread_id’: ‘3’,
 ‘thread_ts’: ‘1f07be97-fd9c-657b-8002-a9eff745d9fc’}}
```


Now, when we resume the stream, the agent will execute our corrected action.


In [ ]:
for event in abot.graph.stream(None, thread):
    for v in event.values():
        print(v)


**Expected output**

```text
Calling: {‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘current weather in Louisiana’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}
Back to the model!
{‘messages’: [ToolMessage(content=”[{‘url’: ‘https://www.weather25.com/north-america/usa/louisiana?page=month&month=August', ‘content’: ‘weather25.com\\nSearch\\nweather in United States\\nRemove from your favorite locations\\nAdd to my locations\\nShare\\nweather in United States\\n\\n# Louisiana weather in August 2025\\n\\nPartly cloudy\\nThundery outbreaks possible\\nPartly cloudy\\nMist\\nPatchy rain possible\\nLight rain shower\\nPatchy rain possible\\nPatchy light rain\\nPatchy rain possible\\nPatchy rain possible\\nOvercast\\nLight rain\\nPatchy rain possible\\nPatchy light rain\\n\\n## The average weather in Louisiana in August […] | 24 Patchy light rain 32° /25° | 25 Patchy rain possible 33° /26° | 26 Patchy rain possible 34° /26° | 27 Overcast 31° /26° | 28 Light rain 27° /25° | 29 Patchy rain possible 27° /25° | 30 Patchy light rain 33° /26° |\\n| 31 Light rain shower 31° /25° | | | | | | | […] | 10 Patchy rain possible 34° /26° | 11 Partly cloudy 34° /26° | 12 Moderate or heavy rain shower 34° /26° | 13 Patchy rain possible 34° /26° | 14 Patchy rain possible 34° /26° | 15 Moderate or heavy rain shower 35° /26° | 16 Patchy rain possible 35° /26° |\\n| 17 Partly cloudy 38° /28° | 18 Thundery outbreaks possible 36° /27° | 19 Partly cloudy 35° /27° | 20 Mist 38° /26° | 21 Patchy rain possible 34° /26° | 22 Light rain shower 32° /25° | 23 Patchy rain possible 32° /25° |’}, {‘url’: ‘https://www.louisiana.gov/', ‘content’: ‘more. […] more. […] # Did you know?’}]”, name=’tavily_search_results_json’, id=’d356f562–1d89–47c5–866e-b506f392a3b9', tool_call_id=’call_6ED1ZQ8nrjYIOY14yqInLPZc’)]}
{‘messages’: [HumanMessage(content=’Whats the weather in LA?’, id=’abe1c1c0-f305–479d-b788–820269a138ce’), AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘function’: {‘arguments’: ‘{“query”:”weather in Los Angeles”}’, ‘name’: ‘tavily_search_results_json’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’, ‘type’: ‘function’}]}, response_metadata={‘finish_reason’: ‘tool_calls’, ‘logprobs’: None, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘token_usage’: {‘completion_tokens’: 22, ‘completion_tokens_details’: {‘accepted_prediction_tokens’: 0, ‘audio_tokens’: 0, ‘reasoning_tokens’: 0, ‘rejected_prediction_tokens’: 0}, ‘prompt_tokens’: 152, ‘prompt_tokens_details’: {‘audio_tokens’: 0, ‘cached_tokens’: 0}, ‘total_tokens’: 174}}, id=’run-7ac75989–7448–4d63-a807–49a0061317ae-0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘current weather in Louisiana’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}]), ToolMessage(content=”[{‘url’: ‘https://www.weather25.com/north-america/usa/louisiana?page=month&month=August', ‘content’: ‘weather25.com\\nSearch\\nweather in United States\\nRemove from your favorite locations\\nAdd to my locations\\nShare\\nweather in United States\\n\\n# Louisiana weather in August 2025\\n\\nPartly cloudy\\nThundery outbreaks possible\\nPartly cloudy\\nMist\\nPatchy rain possible\\nLight rain shower\\nPatchy rain possible\\nPatchy light rain\\nPatchy rain possible\\nPatchy rain possible\\nOvercast\\nLight rain\\nPatchy rain possible\\nPatchy light rain\\n\\n## The average weather in Louisiana in August […] | 24 Patchy light rain 32° /25° | 25 Patchy rain possible 33° /26° | 26 Patchy rain possible 34° /26° | 27 Overcast 31° /26° | 28 Light rain 27° /25° | 29 Patchy rain possible 27° /25° | 30 Patchy light rain 33° /26° |\\n| 31 Light rain shower 31° /25° | | | | | | | […] | 10 Patchy rain possible 34° /26° | 11 Partly cloudy 34° /26° | 12 Moderate or heavy rain shower 34° /26° | 13 Patchy rain possible 34° /26° | 14 Patchy rain possible 34° /26° | 15 Moderate or heavy rain shower 35° /26° | 16 Patchy rain possible 35° /26° |\\n| 17 Partly cloudy 38° /28° | 18 Thundery outbreaks possible 36° /27° | 19 Partly cloudy 35° /27° | 20 Mist 38° /26° | 21 Patchy rain possible 34° /26° | 22 Light rain shower 32° /25° | 23 Patchy rain possible 32° /25° |’}, {‘url’: ‘https://www.louisiana.gov/', ‘content’: ‘more. […] more. […] # Did you know?’}]”, name=’tavily_search_results_json’, id=’d356f562–1d89–47c5–866e-b506f392a3b9', tool_call_id=’call_6ED1ZQ8nrjYIOY14yqInLPZc’), AIMessage(content=’I retrieved information about the weather in Louisiana. It seems to be partly cloudy with thundery outbreaks possible. The temperature ranges from 32° to 25°. If you meant Los Angeles, could you please confirm so I can provide you with the accurate weather information?’, response_metadata={‘token_usage’: {‘completion_tokens’: 56, ‘prompt_tokens’: 656, ‘total_tokens’: 712, ‘prompt_tokens_details’: {‘cached_tokens’: 0, ‘audio_tokens’: 0}, ‘completion_tokens_details’: {‘reasoning_tokens’: 0, ‘audio_tokens’: 0, ‘accepted_prediction_tokens’: 0, ‘rejected_prediction_tokens’: 0}}, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘finish_reason’: ‘stop’, ‘logprobs’: None}, id=’run-ecee6f99-c1bb-4293–8388–602808c7f9ea-0')]}
{‘messages’: [AIMessage(content=’I retrieved information about the weather in Louisiana. It seems to be partly cloudy with thundery outbreaks possible. The temperature ranges from 32° to 25°. If you meant Los Angeles, could you please confirm so I can provide you with the accurate weather information?’, response_metadata={‘token_usage’: {‘completion_tokens’: 56, ‘prompt_tokens’: 656, ‘total_tokens’: 712, ‘prompt_tokens_details’: {‘cached_tokens’: 0, ‘audio_tokens’: 0}, ‘completion_tokens_details’: {‘reasoning_tokens’: 0, ‘audio_tokens’: 0, ‘accepted_prediction_tokens’: 0, ‘rejected_prediction_tokens’: 0}}, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘finish_reason’: ‘stop’, ‘logprobs’: None}, id=’run-ecee6f99-c1bb-4293–8388–602808c7f9ea-0')]}
```


The agent now correctly searches for and reports the weather in Louisiana. We’ve successfully corrected the agent’s course mid-flight!

### 3.2. Time Travel: Rewinding and Branching

Every modification we make creates a new state in history. This means we can always go back to a previous point and explore a different path. This is “time travel.”

Let’s retrieve the entire history for our thread.


In [ ]:
states = []
for state in abot.graph.get_state_history(thread):
    print(state)
    print('--')
    states.append(state)


**Expected output**

```text
StateSnapshot(values={‘messages’: [HumanMessage(content=’Whats the weather in LA?’, id=’abe1c1c0-f305–479d-b788–820269a138ce’), AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘function’: {‘arguments’: ‘{“query”:”weather in Los Angeles”}’, ‘name’: ‘tavily_search_results_json’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’, ‘type’: ‘function’}]}, response_metadata={‘finish_reason’: ‘tool_calls’, ‘logprobs’: None, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘token_usage’: {‘completion_tokens’: 22, ‘completion_tokens_details’: {‘accepted_prediction_tokens’: 0, ‘audio_tokens’: 0, ‘reasoning_tokens’: 0, ‘rejected_prediction_tokens’: 0}, ‘prompt_tokens’: 152, ‘prompt_tokens_details’: {‘audio_tokens’: 0, ‘cached_tokens’: 0}, ‘total_tokens’: 174}}, id=’run-7ac75989–7448–4d63-a807–49a0061317ae-0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘current weather in Louisiana’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}]), ToolMessage(content=”[{‘url’: ‘https://www.weather25.com/north-america/usa/louisiana?page=month&month=August', ‘content’: ‘weather25.com\\nSearch\\nweather in United States\\nRemove from your favorite locations\\nAdd to my locations\\nShare\\nweather in United States\\n\\n# Louisiana weather in August 2025\\n\\nPartly cloudy\\nThundery outbreaks possible\\nPartly cloudy\\nMist\\nPatchy rain possible\\nLight rain shower\\nPatchy rain possible\\nPatchy light rain\\nPatchy rain possible\\nPatchy rain possible\\nOvercast\\nLight rain\\nPatchy rain possible\\nPatchy light rain\\n\\n## The average weather in Louisiana in August […] | 24 Patchy light rain 32° /25° | 25 Patchy rain possible 33° /26° | 26 Patchy rain possible 34° /26° | 27 Overcast 31° /26° | 28 Light rain 27° /25° | 29 Patchy rain possible 27° /25° | 30 Patchy light rain 33° /26° |\\n| 31 Light rain shower 31° /25° | | | | | | | […] | 10 Patchy rain possible 34° /26° | 11 Partly cloudy 34° /26° | 12 Moderate or heavy rain shower 34° /26° | 13 Patchy rain possible 34° /26° | 14 Patchy rain possible 34° /26° | 15 Moderate or heavy rain shower 35° /26° | 16 Patchy rain possible 35° /26° |\\n| 17 Partly cloudy 38° /28° | 18 Thundery outbreaks possible 36° /27° | 19 Partly cloudy 35° /27° | 20 Mist 38° /26° | 21 Patchy rain possible 34° /26° | 22 Light rain shower 32° /25° | 23 Patchy rain possible 32° /25° |’}, {‘url’: ‘https://www.louisiana.gov/', ‘content’: ‘more. […] more. […] # Did you know?’}]”, name=’tavily_search_results_json’, id=’d356f562–1d89–47c5–866e-b506f392a3b9', tool_call_id=’call_6ED1ZQ8nrjYIOY14yqInLPZc’), AIMessage(content=’I retrieved information about the weather in Louisiana. It seems to be partly cloudy with thundery outbreaks possible. The temperature ranges from 32° to 25°. If you meant Los Angeles, could you please confirm so I can provide you with the accurate weather information?’, response_metadata={‘finish_reason’: ‘stop’, ‘logprobs’: None, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘token_usage’: {‘completion_tokens’: 56, ‘completion_tokens_details’: {‘accepted_prediction_tokens’: 0, ‘audio_tokens’: 0, ‘reasoning_tokens’: 0, ‘rejected_prediction_tokens’: 0}, ‘prompt_tokens’: 656, ‘prompt_tokens_details’: {‘audio_tokens’: 0, ‘cached_tokens’: 0}, ‘total_tokens’: 712}}, id=’run-ecee6f99-c1bb-4293–8388–602808c7f9ea-0')]}, next=(), config={‘configurable’: {‘thread_id’: ‘3’, ‘thread_ts’: ‘1f07be98–3632–646e-8004-cccbd173276b’}}, metadata={‘source’: ‘loop’, ‘step’: 4, ‘writes’: {‘llm’: {‘messages’: [AIMessage(content=’I retrieved information about the weather in Louisiana. It seems to be partly cloudy with thundery outbreaks possible. The temperature ranges from 32° to 25°. If you meant Los Angeles, could you please confirm so I can provide you with the accurate weather information?’, response_metadata={‘finish_reason’: ‘stop’, ‘logprobs’: None, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘token_usage’: {‘completion_tokens’: 56, ‘completion_tokens_details’: {‘accepted_prediction_tokens’: 0, ‘audio_tokens’: 0, ‘reasoning_tokens’: 0, ‘rejected_prediction_tokens’: 0}, ‘prompt_tokens’: 656, ‘prompt_tokens_details’: {‘audio_tokens’: 0, ‘cached_tokens’: 0}, ‘total_tokens’: 712}}, id=’run-ecee6f99-c1bb-4293–8388–602808c7f9ea-0')]}}}, created_at=’2025–08–18T04:12:11.039431+00:00', parent_config={‘configurable’: {‘thread_id’: ‘3’, ‘thread_ts’: ‘1f07be98–2ef3–6eba-8003–8c64ffad1b31’}})
 — 
StateSnapshot(values={‘messages’: [HumanMessage(content=’Whats the weather in LA?’, id=’abe1c1c0-f305–479d-b788–820269a138ce’), AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘function’: {‘arguments’: ‘{“query”:”weather in Los Angeles”}’, ‘name’: ‘tavily_search_results_json’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’, ‘type’: ‘function’}]}, response_metadata={‘finish_reason’: ‘tool_calls’, ‘logprobs’: None, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘token_usage’: {‘completion_tokens’: 22, ‘completion_tokens_details’: {‘accepted_prediction_tokens’: 0, ‘audio_tokens’: 0, ‘reasoning_tokens’: 0, ‘rejected_prediction_tokens’: 0}, ‘prompt_tokens’: 152, ‘prompt_tokens_details’: {‘audio_tokens’: 0, ‘cached_tokens’: 0}, ‘total_tokens’: 174}}, id=’run-7ac75989–7448–4d63-a807–49a0061317ae-0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘current weather in Louisiana’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}]), ToolMessage(content=”[{‘url’: ‘https://www.weather25.com/north-america/usa/louisiana?page=month&month=August', ‘content’: ‘weather25.com\\nSearch\\nweather in United States\\nRemove from your favorite locations\\nAdd to my locations\\nShare\\nweather in United States\\n\\n# Louisiana weather in August 2025\\n\\nPartly cloudy\\nThundery outbreaks possible\\nPartly cloudy\\nMist\\nPatchy rain possible\\nLight rain shower\\nPatchy rain possible\\nPatchy light rain\\nPatchy rain possible\\nPatchy rain possible\\nOvercast\\nLight rain\\nPatchy rain possible\\nPatchy light rain\\n\\n## The average weather in Louisiana in August […] | 24 Patchy light rain 32° /25° | 25 Patchy rain possible 33° /26° | 26 Patchy rain possible 34° /26° | 27 Overcast 31° /26° | 28 Light rain 27° /25° | 29 Patchy rain possible 27° /25° | 30 Patchy light rain 33° /26° |\\n| 31 Light rain shower 31° /25° | | | | | | | […] | 10 Patchy rain possible 34° /26° | 11 Partly cloudy 34° /26° | 12 Moderate or heavy rain shower 34° /26° | 13 Patchy rain possible 34° /26° | 14 Patchy rain possible 34° /26° | 15 Moderate or heavy rain shower 35° /26° | 16 Patchy rain possible 35° /26° |\\n| 17 Partly cloudy 38° /28° | 18 Thundery outbreaks possible 36° /27° | 19 Partly cloudy 35° /27° | 20 Mist 38° /26° | 21 Patchy rain possible 34° /26° | 22 Light rain shower 32° /25° | 23 Patchy rain possible 32° /25° |’}, {‘url’: ‘https://www.louisiana.gov/', ‘content’: ‘more. […] more. […] # Did you know?’}]”, name=’tavily_search_results_json’, id=’d356f562–1d89–47c5–866e-b506f392a3b9', tool_call_id=’call_6ED1ZQ8nrjYIOY14yqInLPZc’)]}, next=(‘llm’,), config={‘configurable’: {‘thread_id’: ‘3’, ‘thread_ts’: ‘1f07be98–2ef3–6eba-8003–8c64ffad1b31’}}, metadata={‘source’: ‘loop’, ‘step’: 3, ‘writes’: {‘action’: {‘messages’: [ToolMessage(content=”[{‘url’: ‘https://www.weather25.com/north-america/usa/louisiana?page=month&month=August', ‘content’: ‘weather25.com\\nSearch\\nweather in United States\\nRemove from your favorite locations\\nAdd to my locations\\nShare\\nweather in United States\\n\\n# Louisiana weather in August 2025\\n\\nPartly cloudy\\nThundery outbreaks possible\\nPartly cloudy\\nMist\\nPatchy rain possible\\nLight rain shower\\nPatchy rain possible\\nPatchy light rain\\nPatchy rain possible\\nPatchy rain possible\\nOvercast\\nLight rain\\nPatchy rain possible\\nPatchy light rain\\n\\n## The average weather in Louisiana in August […] | 24 Patchy light rain 32° /25° | 25 Patchy rain possible 33° /26° | 26 Patchy rain possible 34° /26° | 27 Overcast 31° /26° | 28 Light rain 27° /25° | 29 Patchy rain possible 27° /25° | 30 Patchy light rain 33° /26° |\\n| 31 Light rain shower 31° /25° | | | | | | | […] | 10 Patchy rain possible 34° /26° | 11 Partly cloudy 34° /26° | 12 Moderate or heavy rain shower 34° /26° | 13 Patchy rain possible 34° /26° | 14 Patchy rain possible 34° /26° | 15 Moderate or heavy rain shower 35° /26° | 16 Patchy rain possible 35° /26° |\\n| 17 Partly cloudy 38° /28° | 18 Thundery outbreaks possible 36° /27° | 19 Partly cloudy 35° /27° | 20 Mist 38° /26° | 21 Patchy rain possible 34° /26° | 22 Light rain shower 32° /25° | 23 Patchy rain possible 32° /25° |’}, {‘url’: ‘https://www.louisiana.gov/', ‘content’: ‘more. […] more. […] # Did you know?’}]”, name=’tavily_search_results_json’, id=’d356f562–1d89–47c5–866e-b506f392a3b9', tool_call_id=’call_6ED1ZQ8nrjYIOY14yqInLPZc’)]}}}, created_at=’2025–08–18T04:12:10.279877+00:00', parent_config={‘configurable’: {‘thread_id’: ‘3’, ‘thread_ts’: ‘1f07be97-fd9c-657b-8002-a9eff745d9fc’}})
 — 
StateSnapshot(values={‘messages’: [HumanMessage(content=’Whats the weather in LA?’, id=’abe1c1c0-f305–479d-b788–820269a138ce’), AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘function’: {‘arguments’: ‘{“query”:”weather in Los Angeles”}’, ‘name’: ‘tavily_search_results_json’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’, ‘type’: ‘function’}]}, response_metadata={‘finish_reason’: ‘tool_calls’, ‘logprobs’: None, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘token_usage’: {‘completion_tokens’: 22, ‘completion_tokens_details’: {‘accepted_prediction_tokens’: 0, ‘audio_tokens’: 0, ‘reasoning_tokens’: 0, ‘rejected_prediction_tokens’: 0}, ‘prompt_tokens’: 152, ‘prompt_tokens_details’: {‘audio_tokens’: 0, ‘cached_tokens’: 0}, ‘total_tokens’: 174}}, id=’run-7ac75989–7448–4d63-a807–49a0061317ae-0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘current weather in Louisiana’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}])]}, next=(‘action’,), config={‘configurable’: {‘thread_id’: ‘3’, ‘thread_ts’: ‘1f07be97-fd9c-657b-8002-a9eff745d9fc’}}, metadata={‘source’: ‘update’, ‘step’: 2, ‘writes’: {‘llm’: {‘messages’: [HumanMessage(content=’Whats the weather in LA?’, id=’abe1c1c0-f305–479d-b788–820269a138ce’), AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘function’: {‘arguments’: ‘{“query”:”weather in Los Angeles”}’, ‘name’: ‘tavily_search_results_json’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’, ‘type’: ‘function’}]}, response_metadata={‘finish_reason’: ‘tool_calls’, ‘logprobs’: None, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘token_usage’: {‘completion_tokens’: 22, ‘completion_tokens_details’: {‘accepted_prediction_tokens’: 0, ‘audio_tokens’: 0, ‘reasoning_tokens’: 0, ‘rejected_prediction_tokens’: 0}, ‘prompt_tokens’: 152, ‘prompt_tokens_details’: {‘audio_tokens’: 0, ‘cached_tokens’: 0}, ‘total_tokens’: 174}}, id=’run-7ac75989–7448–4d63-a807–49a0061317ae-0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘current weather in Louisiana’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}])]}}}, created_at=’2025–08–18T04:12:05.105999+00:00', parent_config={‘configurable’: {‘thread_id’: ‘3’, ‘thread_ts’: ‘1f07be97-d8da-6f42–8001-c76c6f835810’}})
 — 
StateSnapshot(values={‘messages’: [HumanMessage(content=’Whats the weather in LA?’, id=’abe1c1c0-f305–479d-b788–820269a138ce’), AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘function’: {‘arguments’: ‘{“query”:”weather in Los Angeles”}’, ‘name’: ‘tavily_search_results_json’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’, ‘type’: ‘function’}]}, response_metadata={‘finish_reason’: ‘tool_calls’, ‘logprobs’: None, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘token_usage’: {‘completion_tokens’: 22, ‘completion_tokens_details’: {‘accepted_prediction_tokens’: 0, ‘audio_tokens’: 0, ‘reasoning_tokens’: 0, ‘rejected_prediction_tokens’: 0}, ‘prompt_tokens’: 152, ‘prompt_tokens_details’: {‘audio_tokens’: 0, ‘cached_tokens’: 0}, ‘total_tokens’: 174}}, id=’run-7ac75989–7448–4d63-a807–49a0061317ae-0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘weather in Los Angeles’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}])]}, next=(‘action’,), config={‘configurable’: {‘thread_id’: ‘3’, ‘thread_ts’: ‘1f07be97-d8da-6f42–8001-c76c6f835810’}}, metadata={‘source’: ‘loop’, ‘step’: 1, ‘writes’: {‘llm’: {‘messages’: [AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘function’: {‘arguments’: ‘{“query”:”weather in Los Angeles”}’, ‘name’: ‘tavily_search_results_json’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’, ‘type’: ‘function’}]}, response_metadata={‘finish_reason’: ‘tool_calls’, ‘logprobs’: None, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘token_usage’: {‘completion_tokens’: 22, ‘completion_tokens_details’: {‘accepted_prediction_tokens’: 0, ‘audio_tokens’: 0, ‘reasoning_tokens’: 0, ‘rejected_prediction_tokens’: 0}, ‘prompt_tokens’: 152, ‘prompt_tokens_details’: {‘audio_tokens’: 0, ‘cached_tokens’: 0}, ‘total_tokens’: 174}}, id=’run-7ac75989–7448–4d63-a807–49a0061317ae-0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘weather in Los Angeles’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}])]}}}, created_at=’2025–08–18T04:12:01.251900+00:00', parent_config={‘configurable’: {‘thread_id’: ‘3’, ‘thread_ts’: ‘1f07be97-d895–6846–8000-d9ab433c7029’}})
 — 
StateSnapshot(values={‘messages’: [HumanMessage(content=’Whats the weather in LA?’, id=’abe1c1c0-f305–479d-b788–820269a138ce’)]}, next=(‘llm’,), config={‘configurable’: {‘thread_id’: ‘3’, ‘thread_ts’: ‘1f07be97-d895–6846–8000-d9ab433c7029’}}, metadata={‘source’: ‘loop’, ‘step’: 0, ‘writes’: None}, created_at=’2025–08–18T04:12:01.223464+00:00', parent_config={‘configurable’: {‘thread_id’: ‘3’, ‘thread_ts’: ‘1f07be97-d891–604e-bfff-69cbf63b5f04’}})
 — 
StateSnapshot(values={‘messages’: []}, next=(‘__start__’,), config={‘configurable’: {‘thread_id’: ‘3’, ‘thread_ts’: ‘1f07be97-d891–604e-bfff-69cbf63b5f04’}}, metadata={‘source’: ‘input’, ‘step’: -1, ‘writes’: {‘messages’: [HumanMessage(content=’Whats the weather in LA?’)]}}, created_at=’2025–08–18T04:12:01.221630+00:00', parent_config=None)
 —
```


The history is returned in the newest-to-oldest order. Let’s pick a state from before our Louisiana correction — the original state where the agent planned to search for Los Angeles.


In [ ]:
# The exact index may vary, but we're looking for the first interrupt.
to_replay = states[-3]
print(to_replay)


**Expected output**

```text
StateSnapshot(values={‘messages’: [HumanMessage(content=’Whats the weather in LA?’, id=’abe1c1c0-f305–479d-b788–820269a138ce’), AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘function’: {‘arguments’: ‘{“query”:”weather in Los Angeles”}’, ‘name’: ‘tavily_search_results_json’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’, ‘type’: ‘function’}]}, response_metadata={‘finish_reason’: ‘tool_calls’, ‘logprobs’: None, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘token_usage’: {‘completion_tokens’: 22, ‘completion_tokens_details’: {‘accepted_prediction_tokens’: 0, ‘audio_tokens’: 0, ‘reasoning_tokens’: 0, ‘rejected_prediction_tokens’: 0}, ‘prompt_tokens’: 152, ‘prompt_tokens_details’: {‘audio_tokens’: 0, ‘cached_tokens’: 0}, ‘total_tokens’: 174}}, id=’run-7ac75989–7448–4d63-a807–49a0061317ae-0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘weather in Los Angeles’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}])]}, next=(‘action’,), config={‘configurable’: {‘thread_id’: ‘3’, ‘thread_ts’: ‘1f07be97-d8da-6f42–8001-c76c6f835810’}}, metadata={‘source’: ‘loop’, ‘step’: 1, ‘writes’: {‘llm’: {‘messages’: [AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘function’: {‘arguments’: ‘{“query”:”weather in Los Angeles”}’, ‘name’: ‘tavily_search_results_json’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’, ‘type’: ‘function’}]}, response_metadata={‘finish_reason’: ‘tool_calls’, ‘logprobs’: None, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘token_usage’: {‘completion_tokens’: 22, ‘completion_tokens_details’: {‘accepted_prediction_tokens’: 0, ‘audio_tokens’: 0, ‘reasoning_tokens’: 0, ‘rejected_prediction_tokens’: 0}, ‘prompt_tokens’: 152, ‘prompt_tokens_details’: {‘audio_tokens’: 0, ‘cached_tokens’: 0}, ‘total_tokens’: 174}}, id=’run-7ac75989–7448–4d63-a807–49a0061317ae-0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘weather in Los Angeles’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}])]}}}, created_at=’2025–08–18T04:12:01.251900+00:00', parent_config={‘configurable’: {‘thread_id’: ‘3’, ‘thread_ts’: ‘1f07be97-d895–6846–8000-d9ab433c7029’}})
```


If we want to see what would have happened without our intervention, we can simply resume the stream using this past state’s config.


In [ ]:
# Resume from the selected historical state
for event in abot.graph.stream(None, to_replay.config):
    for k, v in event.items():
        print(v)


**Expected output**

```text
Calling: {‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘weather in Los Angeles’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}
Back to the model!
{‘messages’: [ToolMessage(content=”[{‘url’: ‘https://weathershogun.com/weather/usa/ca/los-angeles/451/august/2025-08-18', ‘content’: ‘Monday, August 18, 2025. Los Angeles, CA — Weather Forecast \\n\\n===============\\n\\n☰\\n\\nLos Angeles, CA\\n\\nImage 1: WeatherShogun.com\\n\\nHomeContactBrowse StatesPrivacy PolicyTerms and Conditions\\n\\n°F)°C)\\n\\n❮\\n\\nTodayTomorrowHourly7 days30 daysAugust\\n\\n❯\\n\\nLos Angeles, California Weather: \\n\\nMonday, August 18, 2025\\n\\nDay 84°\\n\\nNight 64°\\n\\nPrecipitation 0 %\\n\\nWind 8 mph\\n\\nUV Index (0–11+)11\\n\\nTuesday\\n\\n Hourly\\n Today\\n Current Air Quality\\n Hourly Air Quality Forecast\\n 7 days\\n 30 days’}, {‘url’: ‘https://en.climate-data.org/north-america/united-states-of-america/california/los-angeles-714829/t/august-8/', ‘content’: ‘| Max. Temperature °C (°F) | 19.5 °C (67.2) °F | 19.4 °C (66.9) °F | 21.4 °C (70.5) °F | 23.3 °C (73.9) °F | 25.2 °C (77.4) °F | 28.1 °C (82.6) °F | 31.3 °C (88.3) °F | 31.9 °C (89.5) °F | 31 °C (87.7) °F | 27.2 °C (81) °F | 23.1 °C (73.5) °F | 18.8 °C (65.9) °F |\\n| Precipitation / Rainfall mm (in) | 84 (3) | 89 (3) | 54 (2) | 19 (0) | 11 (0) | 3 (0) | 2 (0) | 0 (0) | 4 (0) | 17 (0) | 21 (0) | 53 (2) | […] | Humidity(%) | 52% | 57% | 59% | 55% | 56% | 55% | 52% | 49% | 49% | 49% | 46% | 53% |\\n| Rainy days (d) | 4 | 5 | 4 | 2 | 1 | 0 | 0 | 0 | 1 | 2 | 2 | 4 |\\n| avg. Sun hours (hours) | 7.6 | 7.6 | 8.3 | 9.0 | 9.1 | 10.2 | 11.3 | 10.8 | 9.5 | 8.3 | 7.9 | 7.4 | […] ## \\n\\n## \\n\\n# Los Angeles Weather in August\\n\\nAre you planning a holiday with hopefully nice weather in Los Angeles in August 2025? Here you can find all information about the weather in Los Angeles in August:\\n\\n## Los Angeles weather in August\\n\\n| | | | | | |\\n| — — | — — | — — | — — | — — | — — |\\n| | Temperature August | 24.5°C | 76.1°F | | Precipitation / Rainfall August | 0mm | 0 inches |\\n| | Temperature August max. | 31.9°C | 89.5°F |\\n| | Temperature August min. | 18.2°C | 64.8°F |’}]”, name=’tavily_search_results_json’, id=’61621b56–293a-4253-bef8–73c543cc229d’, tool_call_id=’call_6ED1ZQ8nrjYIOY14yqInLPZc’)]}
{‘messages’: [HumanMessage(content=’Whats the weather in LA?’, id=’abe1c1c0-f305–479d-b788–820269a138ce’), AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘function’: {‘arguments’: ‘{“query”:”weather in Los Angeles”}’, ‘name’: ‘tavily_search_results_json’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’, ‘type’: ‘function’}]}, response_metadata={‘finish_reason’: ‘tool_calls’, ‘logprobs’: None, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘token_usage’: {‘completion_tokens’: 22, ‘completion_tokens_details’: {‘accepted_prediction_tokens’: 0, ‘audio_tokens’: 0, ‘reasoning_tokens’: 0, ‘rejected_prediction_tokens’: 0}, ‘prompt_tokens’: 152, ‘prompt_tokens_details’: {‘audio_tokens’: 0, ‘cached_tokens’: 0}, ‘total_tokens’: 174}}, id=’run-7ac75989–7448–4d63-a807–49a0061317ae-0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘weather in Los Angeles’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}]), ToolMessage(content=”[{‘url’: ‘https://weathershogun.com/weather/usa/ca/los-angeles/451/august/2025-08-18', ‘content’: ‘Monday, August 18, 2025. Los Angeles, CA — Weather Forecast \\n\\n===============\\n\\n☰\\n\\nLos Angeles, CA\\n\\nImage 1: WeatherShogun.com\\n\\nHomeContactBrowse StatesPrivacy PolicyTerms and Conditions\\n\\n°F)°C)\\n\\n❮\\n\\nTodayTomorrowHourly7 days30 daysAugust\\n\\n❯\\n\\nLos Angeles, California Weather: \\n\\nMonday, August 18, 2025\\n\\nDay 84°\\n\\nNight 64°\\n\\nPrecipitation 0 %\\n\\nWind 8 mph\\n\\nUV Index (0–11+)11\\n\\nTuesday\\n\\n Hourly\\n Today\\n Current Air Quality\\n Hourly Air Quality Forecast\\n 7 days\\n 30 days’}, {‘url’: ‘https://en.climate-data.org/north-america/united-states-of-america/california/los-angeles-714829/t/august-8/', ‘content’: ‘| Max. Temperature °C (°F) | 19.5 °C (67.2) °F | 19.4 °C (66.9) °F | 21.4 °C (70.5) °F | 23.3 °C (73.9) °F | 25.2 °C (77.4) °F | 28.1 °C (82.6) °F | 31.3 °C (88.3) °F | 31.9 °C (89.5) °F | 31 °C (87.7) °F | 27.2 °C (81) °F | 23.1 °C (73.5) °F | 18.8 °C (65.9) °F |\\n| Precipitation / Rainfall mm (in) | 84 (3) | 89 (3) | 54 (2) | 19 (0) | 11 (0) | 3 (0) | 2 (0) | 0 (0) | 4 (0) | 17 (0) | 21 (0) | 53 (2) | […] | Humidity(%) | 52% | 57% | 59% | 55% | 56% | 55% | 52% | 49% | 49% | 49% | 46% | 53% |\\n| Rainy days (d) | 4 | 5 | 4 | 2 | 1 | 0 | 0 | 0 | 1 | 2 | 2 | 4 |\\n| avg. Sun hours (hours) | 7.6 | 7.6 | 8.3 | 9.0 | 9.1 | 10.2 | 11.3 | 10.8 | 9.5 | 8.3 | 7.9 | 7.4 | […] ## \\n\\n## \\n\\n# Los Angeles Weather in August\\n\\nAre you planning a holiday with hopefully nice weather in Los Angeles in August 2025? Here you can find all information about the weather in Los Angeles in August:\\n\\n## Los Angeles weather in August\\n\\n| | | | | | |\\n| — — | — — | — — | — — | — — | — — |\\n| | Temperature August | 24.5°C | 76.1°F | | Precipitation / Rainfall August | 0mm | 0 inches |\\n| | Temperature August max. | 31.9°C | 89.5°F |\\n| | Temperature August min. | 18.2°C | 64.8°F |’}]”, name=’tavily_search_results_json’, id=’61621b56–293a-4253-bef8–73c543cc229d’, tool_call_id=’call_6ED1ZQ8nrjYIOY14yqInLPZc’), AIMessage(content=’The weather in Los Angeles today is 84°F during the day and 64°F at night. There is no precipitation expected with a wind speed of 8 mph and a UV Index of 11.’, response_metadata={‘token_usage’: {‘completion_tokens’: 42, ‘prompt_tokens’: 1065, ‘total_tokens’: 1107, ‘prompt_tokens_details’: {‘cached_tokens’: 0, ‘audio_tokens’: 0}, ‘completion_tokens_details’: {‘reasoning_tokens’: 0, ‘audio_tokens’: 0, ‘accepted_prediction_tokens’: 0, ‘rejected_prediction_tokens’: 0}}, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘finish_reason’: ‘stop’, ‘logprobs’: None}, id=’run-24679a8e-4713–4847–969a-4fb9f91d5022–0')]}
{‘messages’: [AIMessage(content=’The weather in Los Angeles today is 84°F during the day and 64°F at night. There is no precipitation expected with a wind speed of 8 mph and a UV Index of 11.’, response_metadata={‘token_usage’: {‘completion_tokens’: 42, ‘prompt_tokens’: 1065, ‘total_tokens’: 1107, ‘prompt_tokens_details’: {‘cached_tokens’: 0, ‘audio_tokens’: 0}, ‘completion_tokens_details’: {‘reasoning_tokens’: 0, ‘audio_tokens’: 0, ‘accepted_prediction_tokens’: 0, ‘rejected_prediction_tokens’: 0}}, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘finish_reason’: ‘stop’, ‘logprobs’: None}, id=’run-24679a8e-4713–4847–969a-4fb9f91d5022–0')]}
```


As expected, the agent executes its original plan and searches for the weather in Los Angeles.

### 3.3. Go Back in Time and Edit

We can also combine time travel with state modification to create a new branch in our execution history. Let’s go back to that same to_replay state but modify it before resuming. This time, let’s ask for a specific source, AccuWeather.


In [ ]:
# Modify the tool call in our historical state object
_id = to_replay.values['messages'][-1].tool_calls[0]['id']
to_replay.values['messages'][-1].tool_calls = [{'name': 'tavily_search_results_json',
  'args': {'query': 'current weather in LA, accuweather'},
  'id': _id}]

# Update the state. This creates a new branch from the `to_replay` point.
branch_state = abot.graph.update_state(to_replay.config, to_replay.values)


**Expected output**

```text
{‘messages’: [HumanMessage(content=’Whats the weather in LA?’, id=’abe1c1c0-f305–479d-b788–820269a138ce’), AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘function’: {‘arguments’: ‘{“query”:”weather in Los Angeles”}’, ‘name’: ‘tavily_search_results_json’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’, ‘type’: ‘function’}]}, response_metadata={‘finish_reason’: ‘tool_calls’, ‘logprobs’: None, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘token_usage’: {‘completion_tokens’: 22, ‘completion_tokens_details’: {‘accepted_prediction_tokens’: 0, ‘audio_tokens’: 0, ‘reasoning_tokens’: 0, ‘rejected_prediction_tokens’: 0}, ‘prompt_tokens’: 152, ‘prompt_tokens_details’: {‘audio_tokens’: 0, ‘cached_tokens’: 0}, ‘total_tokens’: 174}}, id=’run-7ac75989–7448–4d63-a807–49a0061317ae-0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘current weather in LA, accuweather’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}])]}
```


Now, the agent follows this new path, searching for “current weather in LA, accuweather” and providing a result based on that query.


In [ ]:
# Stream from this new branched state
for event in abot.graph.stream(None, branch_state):
    for k, v in event.items():
        if k != "__end__":
            print(v)


**Expected output**

```text
Calling: {‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘current weather in LA, accuweather’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}
Back to the model!
{‘messages’: [ToolMessage(content=”[{‘url’: ‘https://www.accuweather.com/en/us/los-angeles/90012/weather-forecast/347625', ‘content’: ‘Wed\\n\\n8/20\\n\\nPlenty of sunshine\\n\\nMainly clear\\n\\nThu\\n\\n8/21\\n\\nWarm with plenty of sunshine\\n\\nClear\\n\\nFri\\n\\n8/22\\n\\nPlenty of sunshine\\n\\nClear\\n\\nSat\\n\\n8/23\\n\\nPlenty of sunshine\\n\\nClear\\n\\nSun\\n\\n8/24\\n\\nSunshine\\n\\nClear\\n\\nMon\\n\\n8/25\\n\\nPlenty of sunshine\\n\\nClear\\n\\nTue\\n\\n8/26\\n\\nPlenty of sunshine\\n\\nMainly clear\\n\\n## Sun & Moon\\n\\n## Air Quality […] The air has reached a high level of pollution and is unhealthy for sensitive groups. Reduce time spent outside if you are feeling symptoms such as difficulty breathing or throat irritation.\\n\\n## Allergy Outlook\\n\\nTop Stories\\n\\nHurricane\\n\\nHurricane Erin to grow, will next threaten US coasts\\n\\n53 minutes ago\\n\\nWeather Forecasts\\n\\nFall forecast 2025: Warmth to fuel fires, storms before chill hits US\\n\\n7 hours ago\\n\\nHurricane\\n\\nHurricane safety: Explaining rapid intensification and how to prepare […] Tomorrow: Areas of low clouds to start; otherwise, sunny; turning warmer this week\\nHi: 85°\\n\\n## Current Weather\\n\\n9:12 PM\\n\\n## Looking Ahead\\n\\nWarm Thursday\\n\\n## Los Angeles Weather Radar\\n\\nLos Angeles Weather Radar\\n\\n## Hourly Weather\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\n## 10-Day Weather Forecast\\n\\nTonight\\n\\n8/17\\n\\nAreas of low clouds\\n\\nMon\\n\\n8/18\\n\\nMostly sunny\\n\\nNight: Clear\\n\\nTue\\n\\n8/19\\n\\nMostly sunny\\n\\nClear’}, {‘url’: ‘https://www.accuweather.com/en/us/los-angeles/90012/august-weather/347625', ‘content’: ‘# Los Angeles, CA\\n\\nLos Angeles\\n\\nCalifornia\\n\\n## Around the Globe\\n\\nAround the Globe\\n\\n### Hurricane Tracker\\n\\n### Severe Weather\\n\\n### Radar & Maps\\n\\n### News & Features\\n\\n### Astronomy\\n\\n### Business\\n\\n### Climate\\n\\n### Health\\n\\n### Recreation\\n\\n### Sports\\n\\n### Travel\\n\\n### Warnings\\n\\n### Data Suite\\n\\n### Forensics\\n\\n### Advertising\\n\\n### Superior Accuracy™\\n\\n### Video\\n\\n### Winter Center\\n\\n## Monthly\\n\\n## August\\n\\n## 2025\\n\\n## Daily\\n\\n## Temperature Graph\\n\\n## Further Ahead\\n\\nFurther Ahead\\n\\n### September 2025’}]”, name=’tavily_search_results_json’, id=’6a38c9a9–803f-4013–9d6d-c3ad616608da’, tool_call_id=’call_6ED1ZQ8nrjYIOY14yqInLPZc’)]}
{‘messages’: [HumanMessage(content=’Whats the weather in LA?’, id=’abe1c1c0-f305–479d-b788–820269a138ce’), AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘function’: {‘arguments’: ‘{“query”:”weather in Los Angeles”}’, ‘name’: ‘tavily_search_results_json’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’, ‘type’: ‘function’}]}, response_metadata={‘finish_reason’: ‘tool_calls’, ‘logprobs’: None, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘token_usage’: {‘completion_tokens’: 22, ‘completion_tokens_details’: {‘accepted_prediction_tokens’: 0, ‘audio_tokens’: 0, ‘reasoning_tokens’: 0, ‘rejected_prediction_tokens’: 0}, ‘prompt_tokens’: 152, ‘prompt_tokens_details’: {‘audio_tokens’: 0, ‘cached_tokens’: 0}, ‘total_tokens’: 174}}, id=’run-7ac75989–7448–4d63-a807–49a0061317ae-0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘current weather in LA, accuweather’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}]), ToolMessage(content=”[{‘url’: ‘https://www.accuweather.com/en/us/los-angeles/90012/weather-forecast/347625', ‘content’: ‘Wed\\n\\n8/20\\n\\nPlenty of sunshine\\n\\nMainly clear\\n\\nThu\\n\\n8/21\\n\\nWarm with plenty of sunshine\\n\\nClear\\n\\nFri\\n\\n8/22\\n\\nPlenty of sunshine\\n\\nClear\\n\\nSat\\n\\n8/23\\n\\nPlenty of sunshine\\n\\nClear\\n\\nSun\\n\\n8/24\\n\\nSunshine\\n\\nClear\\n\\nMon\\n\\n8/25\\n\\nPlenty of sunshine\\n\\nClear\\n\\nTue\\n\\n8/26\\n\\nPlenty of sunshine\\n\\nMainly clear\\n\\n## Sun & Moon\\n\\n## Air Quality […] The air has reached a high level of pollution and is unhealthy for sensitive groups. Reduce time spent outside if you are feeling symptoms such as difficulty breathing or throat irritation.\\n\\n## Allergy Outlook\\n\\nTop Stories\\n\\nHurricane\\n\\nHurricane Erin to grow, will next threaten US coasts\\n\\n53 minutes ago\\n\\nWeather Forecasts\\n\\nFall forecast 2025: Warmth to fuel fires, storms before chill hits US\\n\\n7 hours ago\\n\\nHurricane\\n\\nHurricane safety: Explaining rapid intensification and how to prepare […] Tomorrow: Areas of low clouds to start; otherwise, sunny; turning warmer this week\\nHi: 85°\\n\\n## Current Weather\\n\\n9:12 PM\\n\\n## Looking Ahead\\n\\nWarm Thursday\\n\\n## Los Angeles Weather Radar\\n\\nLos Angeles Weather Radar\\n\\n## Hourly Weather\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\n## 10-Day Weather Forecast\\n\\nTonight\\n\\n8/17\\n\\nAreas of low clouds\\n\\nMon\\n\\n8/18\\n\\nMostly sunny\\n\\nNight: Clear\\n\\nTue\\n\\n8/19\\n\\nMostly sunny\\n\\nClear’}, {‘url’: ‘https://www.accuweather.com/en/us/los-angeles/90012/august-weather/347625', ‘content’: ‘# Los Angeles, CA\\n\\nLos Angeles\\n\\nCalifornia\\n\\n## Around the Globe\\n\\nAround the Globe\\n\\n### Hurricane Tracker\\n\\n### Severe Weather\\n\\n### Radar & Maps\\n\\n### News & Features\\n\\n### Astronomy\\n\\n### Business\\n\\n### Climate\\n\\n### Health\\n\\n### Recreation\\n\\n### Sports\\n\\n### Travel\\n\\n### Warnings\\n\\n### Data Suite\\n\\n### Forensics\\n\\n### Advertising\\n\\n### Superior Accuracy™\\n\\n### Video\\n\\n### Winter Center\\n\\n## Monthly\\n\\n## August\\n\\n## 2025\\n\\n## Daily\\n\\n## Temperature Graph\\n\\n## Further Ahead\\n\\nFurther Ahead\\n\\n### September 2025’}]”, name=’tavily_search_results_json’, id=’6a38c9a9–803f-4013–9d6d-c3ad616608da’, tool_call_id=’call_6ED1ZQ8nrjYIOY14yqInLPZc’), AIMessage(content=’The current weather in Los Angeles is sunny with a high of 85°F. It is expected to be warm tomorrow with areas of low clouds at the start but turning sunny and warmer throughout the week. If you need more detailed information, I can provide a more specific weather forecast.’, response_metadata={‘token_usage’: {‘completion_tokens’: 57, ‘prompt_tokens’: 821, ‘total_tokens’: 878, ‘prompt_tokens_details’: {‘cached_tokens’: 0, ‘audio_tokens’: 0}, ‘completion_tokens_details’: {‘reasoning_tokens’: 0, ‘audio_tokens’: 0, ‘accepted_prediction_tokens’: 0, ‘rejected_prediction_tokens’: 0}}, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘finish_reason’: ‘stop’, ‘logprobs’: None}, id=’run-f56bbb2b-62c3–47f7-ad7e-6b3f02e5043a-0')]}
{‘messages’: [AIMessage(content=’The current weather in Los Angeles is sunny with a high of 85°F. It is expected to be warm tomorrow with areas of low clouds at the start but turning sunny and warmer throughout the week. If you need more detailed information, I can provide a more specific weather forecast.’, response_metadata={‘token_usage’: {‘completion_tokens’: 57, ‘prompt_tokens’: 821, ‘total_tokens’: 878, ‘prompt_tokens_details’: {‘cached_tokens’: 0, ‘audio_tokens’: 0}, ‘completion_tokens_details’: {‘reasoning_tokens’: 0, ‘audio_tokens’: 0, ‘accepted_prediction_tokens’: 0, ‘rejected_prediction_tokens’: 0}}, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘finish_reason’: ‘stop’, ‘logprobs’: None}, id=’run-f56bbb2b-62c3–47f7-ad7e-6b3f02e5043a-0')]}
```


---

## 4. Manually Adding Tool Results

Finally, let’s look at one of the most powerful HITL patterns: manually providing a tool’s output. This is useful for mocking responses, saving costs on expensive tool calls, or providing information the agent can’t access on its own.

We’ll start from our to_replay state again. This time, instead of modifying the tool call, we’ll add a new ToolMessage to the state, pretending the tool has already run and returned “54 degrees Celsius”.


In [ ]:
_id = to_replay.values['messages'][-1].tool_calls[0]['id']

state_update = {"messages": [ToolMessage(
    tool_call_id=_id,
    name="tavily_search_results_json",
    content="54 degree celcius",
)]}


When we update the state with this new message, we need to tell LangGraph something important. Since we are providing the result of the action node, we need to signal that this node’s work is done. We do this with the as_node=”action” parameter. This tells the graph, “Treat this update as if it came from the action node,” which allows it to correctly determine the next step (which is to go back to the LLM).


In [ ]:
branch_and_add = abot.graph.update_state(
    to_replay.config,
    state_update,
    # This tells the graph to treat this as the output of the 'action' node
    as_node="action")


Now, when we stream from this new state, the agent completely skips the tool call and proceeds directly to the LLM with our mocked data.


In [ ]:
for event in abot.graph.stream(None, branch_and_add):
    for k, v in event.items():
        print(v)


**Expected output**

```text
{‘messages’: [HumanMessage(content=’Whats the weather in LA?’, id=’abe1c1c0-f305–479d-b788–820269a138ce’), AIMessage(content=’’, additional_kwargs={‘tool_calls’: [{‘function’: {‘arguments’: ‘{“query”:”weather in Los Angeles”}’, ‘name’: ‘tavily_search_results_json’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’, ‘type’: ‘function’}]}, response_metadata={‘finish_reason’: ‘tool_calls’, ‘logprobs’: None, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘token_usage’: {‘completion_tokens’: 22, ‘completion_tokens_details’: {‘accepted_prediction_tokens’: 0, ‘audio_tokens’: 0, ‘reasoning_tokens’: 0, ‘rejected_prediction_tokens’: 0}, ‘prompt_tokens’: 152, ‘prompt_tokens_details’: {‘audio_tokens’: 0, ‘cached_tokens’: 0}, ‘total_tokens’: 174}}, id=’run-7ac75989–7448–4d63-a807–49a0061317ae-0', tool_calls=[{‘name’: ‘tavily_search_results_json’, ‘args’: {‘query’: ‘weather in Los Angeles’}, ‘id’: ‘call_6ED1ZQ8nrjYIOY14yqInLPZc’}]), ToolMessage(content=’54 degree celcius’, name=’tavily_search_results_json’, id=’58ddc56e-029d-4ab8-ac64-c2747f7d65a3', tool_call_id=’call_6ED1ZQ8nrjYIOY14yqInLPZc’), AIMessage(content=’The current weather in Los Angeles is 54 degrees Celsius.’, response_metadata={‘token_usage’: {‘completion_tokens’: 13, ‘prompt_tokens’: 190, ‘total_tokens’: 203, ‘prompt_tokens_details’: {‘cached_tokens’: 0, ‘audio_tokens’: 0}, ‘completion_tokens_details’: {‘reasoning_tokens’: 0, ‘audio_tokens’: 0, ‘accepted_prediction_tokens’: 0, ‘rejected_prediction_tokens’: 0}}, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘finish_reason’: ‘stop’, ‘logprobs’: None}, id=’run-99088dda-2b88–46ef-99ab-86c245529ff1–0')]}
{‘messages’: [AIMessage(content=’The current weather in Los Angeles is 54 degrees Celsius.’, response_metadata={‘token_usage’: {‘completion_tokens’: 13, ‘prompt_tokens’: 190, ‘total_tokens’: 203, ‘prompt_tokens_details’: {‘cached_tokens’: 0, ‘audio_tokens’: 0}, ‘completion_tokens_details’: {‘reasoning_tokens’: 0, ‘audio_tokens’: 0, ‘accepted_prediction_tokens’: 0, ‘rejected_prediction_tokens’: 0}}, ‘model_name’: ‘gpt-3.5-turbo’, ‘system_fingerprint’: None, ‘finish_reason’: ‘stop’, ‘logprobs’: None}, id=’run-99088dda-2b88–46ef-99ab-86c245529ff1–0')]}
```


The final response directly uses our injected content: “The current weather in Los Angeles is 54 degrees Celsius.”

---

You now have a comprehensive toolkit for implementing human-in-the-loop workflows with LangGraph. You’ve learned how to:

- Interrupt execution for manual approval.
- Modify the state to correct an agent’s course.
- Use time travel to revisit and branch from past states.
- Inject data to manually guide the agent.

These patterns give you granular control and observability, transforming your agent from a black box into an interactive, steerable collaborator.

So far, we’ve focused on a single-agent architecture. In the final article of this series, we’ll take everything we’ve learned and apply it to a much more complex challenge: building a multi-agent system. See you there.
